In [ ]:
!pip install gensim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# analyse exploratoire de données des images présent dans le dossier
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

image_folder_path = '/content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/input/Flipkart/Images'

# List all files in the directory
image_files = [f for f in os.listdir(image_folder_path) if os.path.isfile(os.path.join(image_folder_path, f))]

print(f"Found {len(image_files)} files in the directory.")

if not image_files:
    print("No image files found in the directory.")
else:
    # Let's analyze some properties of the images
    image_widths = []
    image_heights = []
    image_formats = []

    print("Analyzing image properties (sampling the first 10 images if there are many):")
    # for i, image_file in enumerate(image_files[:10]): # Analyze the first 10 for efficiency
    for i, image_file in enumerate(image_files): # Analyze the first 10 for efficiency
        try:
            img_path = os.path.join(image_folder_path, image_file)
            with Image.open(img_path) as img:
                image_widths.append(img.width)
                image_heights.append(img.height)
                image_formats.append(img.format)
                print(f"  - {image_file}: {img.width}x{img.height}, Format: {img.format}")
        except Exception as e:
            print(f"  - Could not open or read {image_file}: {e}")

    if image_widths and image_heights:
        avg_width = np.mean(image_widths)
        avg_height = np.mean(image_heights)

        print(f"\nAverage image dimensions (based on sampled images): {avg_width:.2f}x{avg_height:.2f}")

        print("\nMost common image formats (based on sampled images):")
        format_counts = {}
        for fmt in image_formats:
            format_counts[fmt] = format_counts.get(fmt, 0) + 1
        for fmt, count in format_counts.items():
            print(f"  - {fmt}: {count}")

        # Display a sample image
        try:
            sample_image_path = os.path.join(image_folder_path, image_files[0])
            print(f"\nDisplaying a sample image: {image_files[0]}")
            img = Image.open(sample_image_path)
            plt.imshow(img)
            plt.axis('off') # Hide axes
            plt.show()
        except Exception as e:
            print(f"Could not display sample image: {e}")
    else:
        print("\nCould not gather image properties from the sampled files.")



In [ ]:
import pandas as pd # Import pandas

print("\nRecherche d'outliers dans les dimensions des images:")

if image_widths and image_heights:
    # Create a DataFrame for image dimensions to easily use describe() and other methods
    image_df = pd.DataFrame({
        'width': image_widths,
        'height': image_heights
    })

    print("\nRésumé statistique des dimensions des images:")
    image_df.describe()

    # Identify potential outliers using the IQR method for image dimensions
    Q1_width = image_df['width'].quantile(0.25)
    Q3_width = image_df['width'].quantile(0.75)
    IQR_width = Q3_width - Q1_width
    lower_bound_width = Q1_width - 1.5 * IQR_width
    upper_bound_width = Q3_width + 1.5 * IQR_width

    Q1_height = image_df['height'].quantile(0.25)
    Q3_height = image_df['height'].quantile(0.75)
    IQR_height = Q3_height - Q1_height
    lower_bound_height = Q1_height - 1.5 * IQR_height
    upper_bound_height = Q3_height + 1.5 * IQR_height

    outliers_width = image_df[(image_df['width'] < lower_bound_width) | (image_df['width'] > upper_bound_width)]
    outliers_height = image_df[(image_df['height'] < lower_bound_height) | (image_df['height'] > upper_bound_height)]

    print(f"\nPotential outliers in image width (using IQR):")
    if not outliers_width.empty:
        # Find the original image files corresponding to these outliers
        outlier_indices_width = outliers_width.index.tolist()
        outlier_files_width = [image_files[i] for i in outlier_indices_width]
        print(f"  Indices: {outlier_indices_width}")
        print(f"  Files: {outlier_files_width}")
        display(outliers_width) # Use display for better formatting
    else:
        print("  Aucun outlier potentiel détecté en largeur selon la méthode IQR.")

    print(f"\nPotential outliers in image height (using IQR):")
    if not outliers_height.empty:
        # Find the original image files corresponding to these outliers
        outlier_indices_height = outliers_height.index.tolist()
        outlier_files_height = [image_files[i] for i in outlier_indices_height]
        print(f"  Indices: {outlier_indices_height}")
        print(f"  Files: {outlier_files_height}")
        display(outliers_height) # Use display for better formatting
    else:
        print("  Aucun outlier potentiel détecté en hauteur selon la méthode IQR.")

    # You can also visualize the distributions to spot outliers more easily
    print("\nVisualisation des distributions des dimensions des images pour repérer les outliers:")

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.boxplot(image_df['width'])
    plt.title('Box Plot des Largeurs des Images')
    plt.ylabel('Largeur (pixels)')

    plt.subplot(1, 2, 2)
    plt.boxplot(image_df['height'])
    plt.title('Box Plot des Hauteurs des Images')
    plt.ylabel('Hauteur (pixels)')

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.scatter(image_df['width'], image_df['height'], alpha=0.5)
    plt.title('Scatter Plot des Dimensions des Images')
    plt.xlabel('Largeur (pixels)')
    plt.ylabel('Hauteur (pixels)')
    plt.grid(True)
    plt.show()


else:
    print("\nImpossible d'analyser les outliers des images car aucune propriété d'image n'a été collectée.")

In [ ]:
image_df.describe()

In [ ]:
# utilise le fichier csv pour l'AED

import pandas as pd


csv_file_path = '/content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/input/Flipkart/flipkart_com-ecommerce_sample_1050.csv'

try:
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(csv_file_path)

    print("\nAnalyse Exploratoire de Données (AED) du DataFrame:")
    print("\nAperçu des 5 premières lignes:")
    display(df.head())

    print("\nInformations sur le DataFrame:")
    df.info()

    print("\nRésumé statistique des colonnes numériques (df.describe()):")
    df.describe()

    # You can now analyze the output of df.describe() to look for potential outliers
    # For example, look at the min/max values and compare them to the mean and percentiles.
    # Extremely large or small values compared to the rest of the distribution could be outliers.

except FileNotFoundError:
    print(f"Erreur: Le fichier CSV n'a pas été trouvé à l'adresse spécifiée: {csv_file_path}")
except Exception as e:
    print(f"Une erreur s'est produite lors de la lecture du fichier CSV: {e}")





In [ ]:
# df['categories'] = df['product_category_tree'].apply(lambda x: x.split('>>'))
# [c.strip("[]'\"").strip() for c in df['product_category_tree'][0].split('>>')]
df['categories'] = df['product_category_tree'].apply(lambda x: [c.strip("[]'\"").strip() for c in x.split('>>')])
# len(df['accurate_category'].unique())
display(df.head())

In [ ]:
# AED sur la colonne 'description'  et détaille la répartition suivant la colonne 'product_category_tree'

print("\nAnalyse univariée de la colonne 'description':")
if 'description' in df.columns:
    # Check for missing values
    missing_descriptions = df['description'].isnull().sum()
    print(f"Nombre de valeurs manquantes dans 'description': {missing_descriptions}")

    # Check unique descriptions
    unique_descriptions = df['description'].nunique()
    print(f"Nombre de descriptions uniques: {unique_descriptions}")

    # Check the distribution of description lengths
    if missing_descriptions < len(df): # Only calculate if there are non-null descriptions
        df['description_length'] = df['description'].str.len().fillna(0)
        print("\nRésumé statistique de la longueur des descriptions:")
        display(df['description_length'].describe())

        # Visualize distribution of description lengths
        plt.figure(figsize=(10, 6))
        df['description_length'].hist(bins=50)
        plt.title('Distribution de la Longueur des Descriptions')
        plt.xlabel('Longueur (nombre de caractères)')
        plt.ylabel('Fréquence')
        plt.show()

        # Look at some of the longest and shortest descriptions (excluding empty/null)
        non_empty_descriptions_df = df[df['description_length'] > 0]
        if not non_empty_descriptions_df.empty:
            print("\nQuelques descriptions les plus longues:")
            for i, row in non_empty_descriptions_df.nlargest(5, 'description_length').iterrows():
                print(f"- Longueur {int(row['description_length'])}: {row['description'][:100]}...") # Print first 100 chars
            print("\nQuelques descriptions les plus courtes (non vides):")
            for i, row in non_empty_descriptions_df.nsmallest(5, 'description_length').iterrows():
                print(f"- Longueur {int(row['description_length'])}: {row['description'][:100]}...")
        else:
            print("\nAucune description non vide à analyser.")

    else:
         print("\nAucune description pour analyser la longueur.")


else:
    print("\nLa colonne 'description' n'existe pas dans le DataFrame.")


print("\nRépartition suivant la colonne 'product_category_tree':")
if 'product_category_tree' in df.columns:
    # Check for missing values
    missing_categories = df['product_category_tree'].isnull().sum()
    print(f"Nombre de valeurs manquantes dans 'product_category_tree': {missing_categories}")

    # Check unique category paths
    unique_category_paths = df['product_category_tree'].nunique()
    print(f"Nombre de chemins de catégorie uniques: {unique_category_paths}")

    # Analyze the distribution of the top-level categories
    # The top-level category is the first element in the list created in the previous step
    df['top_level_category'] = df['categories'].apply(lambda x: x[0] if x else None)

    print("\nDistribution des catégories de premier niveau:")
    top_category_counts = df['top_level_category'].value_counts()
    print(top_category_counts)

    # Visualize the top N top-level categories
    plt.figure(figsize=(12, 7))
    top_category_counts.head(15).plot(kind='bar')
    plt.title('Répartition des 7 Catégories de Premier Niveau les plus Fréquentes')
    plt.xlabel('Catégorie de Premier Niveau')
    plt.ylabel('Nombre de produits')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # You can also explore the distribution of the second level categories if needed
    # df['second_level_category'] = df['categories'].apply(lambda x: x[1] if len(x) > 1 else None)
    # print("\nDistribution des catégories de second niveau:")
    # print(df['second_level_category'].value_counts().head(10)) # Display top 10 second-level categories

else:
    print("\nLa colonne 'product_category_tree' n'existe pas dans le DataFrame.")


In [ ]:
# prompt: approfondis la répartition sur les autres catégories autres que le premier niveau

# Analyse approfondie de la répartition sur les autres catégories au-delà du premier niveau

print("\nAnalyse de la répartition des catégories au-delà du premier niveau:")

if 'categories' in df.columns:
    # Flatten the list of categories for each product
    all_categories = [category for sublist in df['categories'] for category in sublist if category]

    # Count the occurrences of each category at any level
    category_counts = pd.Series(all_categories).value_counts()

    print("\nDistribution des 20 catégories les plus fréquentes (tous niveaux confondus):")
    display(category_counts.head(20))

    # Visualize the top N most frequent categories (all levels)
    plt.figure(figsize=(15, 8))
    category_counts.head(20).plot(kind='bar')
    plt.title('Répartition des 20 Catégories les plus Fréquentes (Tous Niveaux Confondus)')
    plt.xlabel('Catégorie')
    plt.ylabel('Nombre de produits')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # You can also specifically analyze the distribution for the second, third, etc. levels
    # For the second level:
    df['second_level_category'] = df['categories'].apply(lambda x: x[1] if len(x) > 1 else None)
    print("\nDistribution des 15 catégories de second niveau les plus fréquentes:")
    second_level_counts = df['second_level_category'].value_counts()
    display(second_level_counts.head(15))

    plt.figure(figsize=(15, 8))
    second_level_counts.head(15).plot(kind='bar')
    plt.title('Répartition des 15 Catégories de Second Niveau les plus Fréquentes')
    plt.xlabel('Catégorie de Second Niveau')
    plt.ylabel('Nombre de produits')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


    # For the third level (if it exists):
    df['third_level_category'] = df['categories'].apply(lambda x: x[2] if len(x) > 2 else None)
    print("\nDistribution des 15 catégories de troisième niveau les plus fréquentes:")
    third_level_counts = df['third_level_category'].value_counts()
    display(third_level_counts.head(15))

    plt.figure(figsize=(15, 8))
    third_level_counts.head(15).plot(kind='bar')
    plt.title('Répartition des 15 Catégories de Troisième Niveau les plus Fréquentes')
    plt.xlabel('Catégorie de Troisième Niveau')
    plt.ylabel('Nombre de produits')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # You can continue this for deeper levels if needed
else:
    print("\nLa colonne 'categories' n'a pas été correctement préparée.")

In [ ]:
#  approche de type bag-of-words en comptage simple sur la colonne 'description'

from sklearn.feature_extraction.text import CountVectorizer

# Implémenter une approche de type bag-of-words (simple counting) sur la colonne 'description'

# Assurez-vous d'avoir la colonne 'description' et qu'elle n'est pas vide/nulle
if 'description' in df.columns and not df['description'].isnull().all():
    # Remplacer les valeurs NaN par une chaîne vide pour le vectoriseur
    df['description_filled'] = df['description'].fillna('')

    # Créer un vectoriseur de comptage
    # min_df=1: Inclure les mots qui apparaissent dans au moins 1 document (produit)
    # stop_words='english': Supprimer les mots vides courants en anglais.
    #                    Vous pourriez vouloir adapter ceci si votre description est dans une autre langue ou contient des termes spécifiques.
    # token_pattern: Utiliser le modèle par défaut pour diviser en mots (lettres minuscules ou majuscules).
    vectorizer = CountVectorizer(stop_words='english', min_df=1)

    # Adapter et transformer les descriptions en matrice Bag-of-Words
    # Le résultat est une matrice creuse (sparse matrix) pour économiser de la mémoire
    bow_matrix = vectorizer.fit_transform(df['description_filled'])

    print("\nBag-of-Words (Simple Counting) sur la colonne 'description':")
    print(f"Forme de la matrice BoW: {bow_matrix.shape}") # (nombre de documents, nombre de mots uniques)
    print(f"Nombre de mots uniques (features): {bow_matrix.shape[1]}")

    # Vous pouvez obtenir la liste des mots (features)
    feature_names = vectorizer.get_feature_names_out()
    print(f"\nQuelques exemples de mots (features): {feature_names[:20]}...")

    # Vous pouvez inspecter la matrice BoW pour quelques documents
    # print("\nMatrice BoW pour les 5 premières descriptions (peut être volumineux):")
    # print(bow_matrix[:5].todense()) # Convertir en matrice dense pour l'affichage

    # Afficher la somme des fréquences de chaque mot (pour voir les mots les plus fréquents)
    word_counts = bow_matrix.sum(axis=0)
    word_freq = [(word, word_counts[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
    word_freq = sorted(word_freq, key=lambda x: x[1], reverse=True)

    print("\nLes 20 mots les plus fréquents dans les descriptions:")
    for word, count in word_freq[:50]:
        print(f"- {word}: {count}")

    # Si vous avez besoin de la matrice dense pour certaines opérations (attention à la mémoire pour les grands datasets)
    # bow_dense_matrix = bow_matrix.todense()

else:
    print("\nLa colonne 'description' n'est pas disponible ou est vide dans le DataFrame pour l'analyse Bag-of-Words.")



In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text

# Download the WordNet corpus for lemmatization if you haven't already
nltk.download('wordnet')

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Get words with more than 350 occurrences
frequent_words = [word for word, count in word_freq if count > 350]

# Lemmatize the frequent words
lemmatized_frequent_words = [lemmatizer.lemmatize(word) for word in frequent_words]

# Create a custom stopword list by extending the default English stopwords
# with both the original and lemmatized versions of the frequent words
custom_stop_words_frequent = text.ENGLISH_STOP_WORDS.union(frequent_words, lemmatized_frequent_words)

# Re-run the CountVectorizer with the new custom stopword list
vectorizer_frequent = CountVectorizer(stop_words=list(custom_stop_words_frequent), min_df=1)
bow_matrix_frequent = vectorizer_frequent.fit_transform(df['description_filled'])

print("\nBag-of-Words with lemmatized frequent stopwords (occ > 225):")
print(f"Shape of the new BoW matrix: {bow_matrix_frequent.shape}")

# Display the new most frequent words to confirm the old ones were removed
word_counts_frequent = bow_matrix_frequent.sum(axis=0)
word_freq_frequent = [(word, word_counts_frequent[0, idx]) for word, idx in vectorizer_frequent.vocabulary_.items()]
word_freq_frequent = sorted(word_freq_frequent, key=lambda x: x[1], reverse=True)

print("\nThe 50 most frequent words after adding frequent lemmatized stopwords:")
for word, count in word_freq_frequent[:50]:
    print(f"- {word}: {count}")

In [ ]:
!pip install wordcloud
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def generate_wordcloud(word_frequencies, stopwords):
    """
    Generates and displays a word cloud from a dictionary of word frequencies.

    Args:
        word_frequencies (dict): A dictionary where keys are words and values are their frequencies.
        stopwords (set): A set of words to exclude from the word cloud.
    """
    wordcloud = WordCloud(width = 800, height = 800,
                    background_color ='white',
                    stopwords = stopwords,
                    min_font_size = 10).generate_from_frequencies(word_frequencies)

    # Plot the WordCloud image
    plt.figure(figsize = (8, 8), facecolor = None)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.tight_layout(pad = 0)

    plt.show()

# Generate the word cloud using the new function
generate_wordcloud(dict(word_freq_frequent), set(custom_stop_words_frequent))

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text

# Add the new words to the stopword list
additional_stopwords = ['213', 'inch', 'men', 'india', 'size', 'dimensions', 'color', 'girl', 'warranty', 'details', 'brand', 'general', 'quality', 'best', 'model', 'pack', 'package', 'prices', 'sales', 'great', 'baby', 'type', 'set', 'analog', 'material', 'cotton', 'box', 'key', 'number', 'contents', 'design', 'specifications', 'ceramic']
lemmatized_additional_stopwords = [lemmatizer.lemmatize(word) for word in additional_stopwords]

custom_stop_words_updated = custom_stop_words_frequent.union(additional_stopwords, lemmatized_additional_stopwords)

# Re-run the CountVectorizer with the updated stopword list
vectorizer_updated = CountVectorizer(stop_words=list(custom_stop_words_updated), min_df=1)
bow_matrix_updated = vectorizer_updated.fit_transform(df['description_filled'])

print("\nBag-of-Words with updated lemmatized stopwords:")
print(f"Shape of the new BoW matrix: {bow_matrix_updated.shape}")

# Display the new most frequent words
word_counts_updated = bow_matrix_updated.sum(axis=0)
word_freq_updated = [(word, word_counts_updated[0, idx]) for word, idx in vectorizer_updated.vocabulary_.items()]
word_freq_updated = sorted(word_freq_updated, key=lambda x: x[1], reverse=True)

print("\nThe 20 most frequent words after updating stopwords:")
for word, count in word_freq_updated[:20]:
    print(f"- {word}: {count}")

In [ ]:
generate_wordcloud(dict(word_freq_updated), set(custom_stop_words_updated))

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

def ARI_fct(true_labels, cluster_labels, tsne_results, title_suffix=""):
    """
    Calculates ARI score and visualizes t-SNE results colored by true labels and clusters.

    Args:
        true_labels (array-like): The true labels for each data point.
        cluster_labels (array-like): The predicted cluster labels for each data point.
        tsne_results (array-like): The 2D t-SNE results (shape: n_samples, 2).
        title_suffix (str): A suffix to add to the plot titles.
    """
    # Calculate the Adjusted Rand Index (ARI)
    ari_score = adjusted_rand_score(true_labels, cluster_labels)
    print(f"Adjusted Rand Index (ARI) score{title_suffix}: {ari_score:.4f}")

    # Create a DataFrame for the t-SNE results
    tsne_df = pd.DataFrame(data=tsne_results, columns=['tsne-1', 'tsne-2'])
    tsne_df['cluster'] = cluster_labels
    tsne_df['category'] = true_labels

    # Determine the number of unique categories/clusters for plotting
    n_categories = len(tsne_df['category'].unique())
    n_clusters_plot = len(tsne_df['cluster'].unique())

    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

    # Visualize the t-SNE results, colored by cluster
    sns.scatterplot(
        x="tsne-1", y="tsne-2",
        hue="cluster",
        palette=sns.color_palette("hsv", n_clusters_plot),
        data=tsne_df,
        legend="full",
        alpha=0.7,
        ax=ax1
    )
    ax1.set_title(f't-SNE visualization, colored by K-Means cluster{title_suffix}')
    ax1.set_xlabel('t-SNE Component 1')
    ax1.set_ylabel('t-SNE Component 2')
    ax1.legend(title='Cluster')

    # Visualize the t-SNE results, colored by true category
    sns.scatterplot(
        x="tsne-1", y="tsne-2",
        hue="category",
        palette=sns.color_palette("hsv", n_categories),
        data=tsne_df,
        legend="full",
        alpha=0.7,
        ax=ax2
    )
    ax2.set_title(f't-SNE visualization, colored by true category{title_suffix}')
    ax2.set_xlabel('t-SNE Component 1')
    ax2.set_ylabel('t-SNE Component 2')
    ax2.legend(title='Category')

    plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
import numpy as np

# Apply K-Means clustering on the simple count BoW matrix
n_clusters = df['top_level_category'].nunique()
kmeans_bow = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels_bow = kmeans_bow.fit_predict(bow_matrix_updated)

# Apply t-SNE on the simple count BoW matrix
tsne_bow = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=300, init='random')
tsne_results_bow = tsne_bow.fit_transform(bow_matrix_updated)

# Call the ARI_fct to calculate ARI and visualize
ARI_fct(df['top_level_category'], cluster_labels_bow, tsne_results_bow, title_suffix=" (Simple Count BoW Features)")

In [ ]:
#  approche de type bag-of-words en comptage Tf-idf sur la colonne 'description'

from sklearn.feature_extraction.text import TfidfVectorizer

# Implémenter une approche de type Bag-of-Words en comptage Tf-idf sur la colonne 'description'

# Assurez-vous d'avoir la colonne 'description' et qu'elle n'est pas vide/nulle
if 'description' in df.columns and not df['description'].isnull().all():
    # Remplacer les valeurs NaN par une chaîne vide pour le vectoriseur
    df['description_filled'] = df['description'].fillna('')

    # Créer un vectoriseur TF-IDF
    # min_df=1: Inclure les mots qui apparaissent dans au moins 1 document (produit)
    # stop_words='english': Supprimer les mots vides courants en anglais.
    #                     Vous pourriez vouloir adapter ceci si votre description est dans une autre langue ou contient des termes spécifiques.
    # token_pattern: Utiliser le modèle par défaut pour diviser en mots.
    # use_idf=True: Activer le calcul TF-IDF (c'est le comportement par défaut de TfidfVectorizer, mais explicite)
    # smooth_idf=True: Ajoute 1 à la fréquence du document pour éviter les divisions par zéro (utile pour les termes rares)
    # norm='l2': Normalise chaque vecteur TF-IDF à une norme L2 de 1 (unité de longueur).

    tfidf_vectorizer = TfidfVectorizer(stop_words='english', min_df=1, use_idf=True, smooth_idf=True, norm='l2')

    # Adapter et transformer les descriptions en matrice TF-IDF
    # Le résultat est une matrice creuse (sparse matrix)
    tfidf_matrix = tfidf_vectorizer.fit_transform(df['description_filled'])

    print("\nTF-IDF Bag-of-Words sur la colonne 'description':")
    print(f"Forme de la matrice TF-IDF: {tfidf_matrix.shape}") # (nombre de documents, nombre de mots uniques)
    print(f"Nombre de mots uniques (features): {tfidf_matrix.shape[1]}")

    # Vous pouvez obtenir la liste des mots (features)
    tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
    print(f"\nQuelques exemples de mots (features) du TF-IDF: {tfidf_feature_names[:20]}...")

    # Vous pouvez inspecter la matrice TF-IDF pour quelques documents
    # print("\nMatrice TF-IDF pour les 5 premières descriptions (peut être volumineux):")
    # print(tfidf_matrix[:5].todense()) # Convertir en matrice dense pour l'affichage

    # Pour voir les termes les plus importants dans une description spécifique (basé sur leur score TF-IDF)
    # Choisissez un index de document (par exemple, le premier document)
    document_index = 0
    if document_index < tfidf_matrix.shape[0]:
        tfidf_scores = tfidf_matrix[document_index].toarray().flatten() # Convertir en array 1D
        feature_scores = list(zip(tfidf_feature_names, tfidf_scores))
        # Trier par score TF-IDF décroissant
        sorted_features = sorted(feature_scores, key=lambda x: x[1], reverse=True)

        print(f"\nLes 10 mots les plus importants (TF-IDF) pour le document {document_index}:")
        for word, score in sorted_features[:10]:
            print(f"- {word}: {score:.4f}")

    # Si vous avez besoin de la matrice dense TF-IDF pour certaines opérations (attention à la mémoire pour les grands datasets)
    # tfidf_dense_matrix = tfidf_matrix.todense()

else:
    print("\nLa colonne 'description' n'est pas disponible ou est vide dans le DataFrame pour l'analyse TF-IDF.")


In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text

# Add the new words to the stopword list
additional_stopwords += ['213', 'polyester', 'abstract', 'multicolor', 'ideal', 'women', 'number', 'width', 'discounts', 'printed']
lemmatized_additional_stopwords = [lemmatizer.lemmatize(word) for word in additional_stopwords]

custom_stop_words_updated = custom_stop_words_updated.union(additional_stopwords, lemmatized_additional_stopwords)

# Re-run the CountVectorizer with the updated stopword list
vectorizer_updated = CountVectorizer(stop_words=list(custom_stop_words_updated), min_df=1)
bow_matrix_updated = vectorizer_updated.fit_transform(df['description_filled'])

print("\nBag-of-Words with updated lemmatized stopwords:")
print(f"Shape of the new BoW matrix: {bow_matrix_updated.shape}")

# Display the new most frequent words
word_counts_updated = bow_matrix_updated.sum(axis=0)
word_freq_updated = [(word, word_counts_updated[0, idx]) for word, idx in vectorizer_updated.vocabulary_.items()]
word_freq_updated = sorted(word_freq_updated, key=lambda x: x[1], reverse=True)

print("\nThe 20 most frequent words after updating stopwords:")
for word, count in word_freq_updated[:20]:
    print(f"- {word}: {count}")

In [ ]:
len(custom_stop_words_updated)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
import numpy as np

# Apply K-Means clustering on the TF-IDF matrix
n_clusters = df['top_level_category'].nunique()
kmeans_tfidf = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels_tfidf = kmeans_tfidf.fit_predict(tfidf_matrix)

# Apply t-SNE on the TF-IDF matrix
tsne_tfidf = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=300, init='random')
tsne_results_tfidf = tsne_tfidf.fit_transform(tfidf_matrix)

# Call the ARI_fct to calculate ARI and visualize
ARI_fct(df['top_level_category'], cluster_labels_tfidf, tsne_results_tfidf, title_suffix=" (TF-IDF Features)")

# Word2vec

In [ ]:
# prompt: met en oeuvre une approche de type word/sentence embedding classique avec Word2Vec (ou Glove ou FastText)  sur la colonne 'description'
# !pip install gensim # Removed to avoid potential conflicts after restart


from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Add this line to download the missing resource
from nltk.tokenize import word_tokenize
import pandas as pd # Ensure pandas is imported
import numpy as np # Ensure numpy is imported
import matplotlib.pyplot as plt # Ensure matplotlib is imported
from sklearn.manifold import TSNE # Ensure TSNE is imported


# Ensure 'description' column exists and fill NaNs
if 'description' in df.columns:
    df['description_filled'] = df['description'].fillna('')

    # Tokenize the descriptions and remove stopwords
    # Use NLTK's word_tokenize for better tokenization than simple split()
    # Convert to lowercase and remove stopwords
    df['tokenized_description'] = df['description_filled'].apply(lambda x: [word for word in word_tokenize(x.lower()) if word not in custom_stop_words_updated])


    print("\nExemple de description tokenisée (avec suppression des stopwords):")
    display(df['tokenized_description'].head())

    # Train a Word2Vec model
    # We need a list of lists of words (sentences)
    sentences = df['tokenized_description'].tolist()

    # Filter out empty sentences that could cause issues with Word2Vec
    sentences = [s for s in sentences if s]


    # Parameters for Word2Vec:
    # vector_size: dimension of word vectors (embeddings)
    # window: maximum distance between the current and predicted word within a sentence.
    # min_count: ignores all words with total frequency lower than this.
    # workers: use these many worker threads to train the model (=faster training with multicore machines).
    # sg: 1 for skip-gram, 0 for CBOW. Skip-gram is usually better for rare words.

    # Adjust parameters based on your dataset size and desired performance
    if sentences: # Only train if there are valid sentences
        word2vec_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4, sg=0)

        # The model is trained, you can now access word vectors
        # For example, get the vector for a specific word:
        # try:
        #     vector = word2vec_model.wv['mobile']
        #     print(f"\nVector for 'mobile': {vector[:10]}...") # Print first 10 dimensions
        # except KeyError:
        #     print("\n'mobile' not in vocabulary.")

        # You can also find similar words
        # try:
        #     similar_words = word2vec_model.wv.most_similar('mobile')
        #     print(f"\nWords similar to 'mobile': {similar_words}")
        # except KeyError:
        #      print("\n'mobile' not in vocabulary.")


        # Generate Sentence/Document Embeddings
        # A common approach is to average the word vectors in a document.
        # This is a simple baseline; more complex methods exist (e.g., training a Doc2Vec model)

        def get_document_embedding(tokens, model):
            # Filter out words not in the model's vocabulary
            valid_words = [word for word in tokens if word in model.wv]
            if not valid_words:
                return None # Or a vector of zeros if preferred
            # Average the word vectors
            return np.mean([model.wv[word] for word in valid_words], axis=0)

        print("\nGenerating document embeddings by averaging word vectors...")
        # Apply the function to create document embeddings
        df['document_embedding'] = df['tokenized_description'].apply(lambda x: get_document_embedding(x, word2vec_model))

        # Remove rows where embedding could not be generated (e.g., empty description)
        df_with_embeddings = df.dropna(subset=['document_embedding']).copy()

        print(f"Nombre de documents avec des embeddings générés: {len(df_with_embeddings)}")

        # The 'document_embedding' column now contains the vector representation for each description
        # Each entry is a numpy array of shape (vector_size,)
        print("\nExemple de document embedding (pour la première description avec embedding):")
        if not df_with_embeddings.empty:
            print(df_with_embeddings.iloc[0]['document_embedding'][:10]) # Print first 10 dimensions
            print(f"Shape de l'embedding: {df_with_embeddings.iloc[0]['document_embedding'].shape}")
        else:
            print("Aucun embedding de document généré.")
    else:
        print("\nImpossible d'entraîner le modèle Word2Vec car il n'y a pas de phrases valides après la tokenisation.")


else:
    print("\nLa colonne 'description' n'est pas disponible dans le DataFrame.")

This code is calculating the Adjusted Rand Index (ARI) to see how well the clusters found by K-Means match the actual product categories, and it is also generating two t-SNE plots to visualize the results: one colored by the predicted clusters and one colored by the true categories. This allows us to both quantitatively and qualitatively evaluate the performance of our Word2Vec-based clustering.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
import numpy as np

# Apply K-Means clustering on the Word2Vec document embeddings
n_clusters = df_with_embeddings['top_level_category'].nunique()
kmeans_w2v = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels_w2v = kmeans_w2v.fit_predict(np.vstack(df_with_embeddings['document_embedding'].values))

# Apply t-SNE on the Word2Vec document embeddings
tsne_w2v = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=300, init='pca')
tsne_results_w2v = tsne_w2v.fit_transform(np.vstack(df_with_embeddings['document_embedding'].values))

# Call the ARI_fct to calculate ARI and visualize
ARI_fct(df_with_embeddings['top_level_category'], cluster_labels_w2v, tsne_results_w2v, title_suffix=" (Word2Vec Features)")

Objective:

The main goal of this project was to explore different methods for automatically classifying products into categories based on their textual descriptions and images. We used a dataset of 1050 products from Flipkart, which included product descriptions, images, and category information.

Methodology:

We explored several techniques for feature extraction and dimensionality reduction, including:

1. Text-based Approaches:

Bag-of-Words (BoW): We started with a simple BoW approach, which represents each product description as a vector of word counts. This method is easy to implement but does not capture the semantic meaning of words.
TF-IDF: We then used TF-IDF (Term Frequency-Inverse Document Frequency) to give more weight to words that are more important in a document. This is an improvement over BoW, but it still does not capture the context of words.
Word2Vec: We used a pre-trained Word2Vec model to generate word embeddings, which are dense vector representations of words that capture their semantic relationships. We then averaged the word embeddings for each description to get a sentence embedding.
FastText: We also used FastText, which is an extension of Word2Vec that can handle out-of-vocabulary words.
BERT: Finally, we used a pre-trained BERT model to generate contextualized word embeddings. This is the most advanced technique we used, and it is expected to provide the best results.
2. Image-based Approaches:

Pixel-based: We started by resizing the images to a fixed size and then flattening them into a vector of pixel values. This is a very simple approach that does not capture the high-level features of the images.
SIFT/ORB: We used SIFT (Scale-Invariant Feature Transform) and ORB (Oriented FAST and Rotated BRIEF) to extract keypoints and descriptors from the images. These features are more robust to changes in scale, rotation, and illumination than raw pixel values.
Bag-of-Visual-Words (BoVW): We used the SIFT descriptors to create a "visual vocabulary" and then represented each image as a histogram of visual words. This is a more advanced technique that can capture the texture and a

In [ ]:
# prompt: implémente à présent une approche bert sur la colonne 'description' puir applique t-sne
%%time
!pip install transformers torch
import torch
from transformers import BertTokenizer, BertModel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load pre-trained model tokenizer (vocabulary)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# Load pre-trained model
model = BertModel.from_pretrained('bert-base-uncased',
                                  output_hidden_states = True, # Whether the model returns all hidden-states.
                                  )

# Put the model in "evaluation" mode, meaning feed-forward operation.
model.eval()

def get_bert_embedding(text, model, tokenizer):
    # Add the special tokens.
    marked_text = "[CLS] " + text + " [SEP]"

    # Split the sentence into tokens.
    tokenized_text = tokenizer.tokenize(marked_text)

    # Truncate the sequence to 512 tokens
    tokenized_text = tokenized_text[:512]

    # Map the token strings to their vocabulary indeces.
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)

    # Mark each of the tokens as belonging to sentence "1".
    segments_ids = [1] * len(tokenized_text)

    # Convert inputs to PyTorch tensors
    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segments_ids])

    # Run the text through BERT, and collect all of the hidden states produced
    # from all 12 layers.
    with torch.no_grad():
        outputs = model(tokens_tensor, segments_tensors)
        hidden_states = outputs[2]

    # `token_vecs` is a tensor with shape [12 x # tokens x 768]
    token_vecs = torch.stack(hidden_states, dim=0)

    # Calculate the average of all token vectors.
    sentence_embedding = torch.mean(token_vecs, dim=0)
    sentence_embedding = torch.mean(sentence_embedding, dim=0)


    return sentence_embedding.numpy()

# Ensure 'description' column exists and fill NaNs
if 'description' in df.columns:
    df['description_filled'] = df['description'].fillna('')

    print("\nGenerating BERT embeddings for descriptions...")
    # This can be slow. Consider running on a GPU and using batching for larger datasets.
    # For demonstration, we'll process a subset.
    num_samples_for_bert = 200 # Adjust as needed, especially if running on CPU
    # df_bert_sample = df.sample(n=num_samples_for_bert, random_state=42).copy()
    # df_bert_sample.reset_index(drop=True, inplace=True)
    df_bert_sample = df.copy()


    df_bert_sample['bert_embedding'] = df_bert_sample['description_filled'].apply(lambda x: get_bert_embedding(x, model, tokenizer))

    # Stack the embeddings into a single numpy array
    bert_embeddings_matrix = np.vstack(df_bert_sample['bert_embedding'].values)

    print(f"Shape of BERT embeddings matrix: {bert_embeddings_matrix.shape}")

    # Apply t-SNE on the BERT embeddings
    print("\nApplying t-SNE on BERT embeddings...")
    tsne_bert = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=300, init='pca')
    tsne_results_bert = tsne_bert.fit_transform(bert_embeddings_matrix)

    # Create a DataFrame for the t-SNE results
    tsne_bert_df = pd.DataFrame(data = tsne_results_bert, columns = ['tsne-bert-one', 'tsne-bert-two'])

    # Add category information for visualization
    tsne_bert_df['top_level_category'] = df_bert_sample['top_level_category']

    print("\nT-SNE results DataFrame for BERT Embeddings head:")
    display(tsne_bert_df.head())

    # Visualize the t-SNE results
    plt.figure(figsize=(12, 10))

    sns.scatterplot(
        x="tsne-bert-one", y="tsne-bert-two",
        hue="top_level_category",
        palette=sns.color_palette("hsv", len(tsne_bert_df['top_level_category'].unique())),
        data=tsne_bert_df,
        legend="full",
        alpha=0.7,
        s=50
    )

    plt.title('T-SNE Visualization of BERT Document Embeddings (Colored by Top Category)')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.legend(title='Top Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

else:
    print("\n'description' column not found in DataFrame. Cannot generate BERT embeddings.")

To apply t-SNE on the images and visualize the clusters, we can follow these steps:

1. **Load and preprocess the images:** We will load the resized images from the /content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/output/Flipkart/Images_resized_median folder, convert them to grayscale, and flatten them into a feature matrix.
2. **Apply t-SNE:** We will then apply t-SNE to this feature matrix to reduce the dimensionality to 2D.
3. **Visualize the results:** Finally, we will create a scatter plot of the t-SNE results, where each point represents an image. We will color the points based on their corresponding product categories to see if the images from the same category form distinct clusters.

Common issues:
The error in the selected cell is a ValueError because the images in your resized_image_folder have different dimensions.

When you flatten the images into 1D arrays, they have different lengths, which causes an error when you try to stack them into a single matrix using np.vstack().

To fix this, we need to resize all the images to a consistent size before flattening them. I will modify the code to resize all images to 224x224 pixels, which is a common size for image classification models. This will ensure that all flattened image arrays have the same length, and the t-SNE can be applied without error.



It's interesting to see that some of the points from the "Computers" category are very far away from the main cluster. This could be due to a few reasons:

- Sub-categories: The "Computers" category is very broad and can contain many sub-categories, such as laptops, desktops, monitors, keyboards, mice, etc. The outliers might belong to a sub-category that is visually very different from the other sub-categories. For example, a mouse will look very different from a laptop.
- Visual diversity: Even within the same sub-category, there can be a lot of visual diversity. For example, a gaming laptop might look very different from a business laptop, even though they both belong to the "laptops" sub-category.
- Incorrect labels: It's also possible that some of the products are mislabeled and don't actually belong to the "Computers" category.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
import numpy as np

# Apply K-Means clustering on the BERT embeddings
# We use the number of unique top-level categories in the sample as the number of clusters
n_clusters_bert = df_bert_sample['top_level_category'].nunique()
kmeans_bert = KMeans(n_clusters=n_clusters_bert, random_state=42, n_init=10)
cluster_labels_bert = kmeans_bert.fit_predict(bert_embeddings_matrix)

# Apply t-SNE on the BERT embeddings
tsne_bert = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=300, init='pca')
tsne_results_bert = tsne_bert.fit_transform(bert_embeddings_matrix)

# Call the ARI_fct to calculate ARI and visualize
# Ensure that the true labels passed to ARI_fct correspond to the same data points as the cluster labels and t-SNE results
ARI_fct(df_bert_sample['top_level_category'], cluster_labels_bert, tsne_results_bert, title_suffix=" (BERT Features)")

In [ ]:
# implémente à présent une approche USE sur la colonne 'description' puis applique t-sne
%%time
!pip install --quiet tensorflow tensorflow-hub
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the Universal Sentence Encoder model
print("\nLoading Universal Sentence Encoder model...")
# You can choose a different USE model if needed (e.g., smaller or larger)
use_model = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
print("Model loaded.")

def get_use_embedding(text, model):
    # Encode the text using the USE model
    # The model expects a list of strings, so we pass the text in a list
    embedding = model([text])
    return embedding.numpy().flatten() # Convert to numpy array and flatten

# Ensure 'description' column exists and fill NaNs
if 'description' in df.columns:
    df['description_filled'] = df['description'].fillna('')

    print("\nGenerating USE embeddings for descriptions...")
    # This can be slow. Consider running on a GPU and using batching for larger datasets.
    # For demonstration, we'll process the entire dataset.
    df['use_embedding'] = df['description_filled'].apply(lambda x: get_use_embedding(x, use_model))

    # Stack the embeddings into a single numpy array
    use_embeddings_matrix = np.vstack(df['use_embedding'].values)

    print(f"Shape of USE embeddings matrix: {use_embeddings_matrix.shape}")

    # Apply t-SNE on the USE embeddings
    print("\nApplying t-SNE on USE embeddings...")
    # Adjust perplexity and n_iter as needed for better visualization
    tsne_use = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300, init='pca')
    tsne_results_use = tsne_use.fit_transform(use_embeddings_matrix)

    # Create a DataFrame for the t-SNE results
    tsne_use_df = pd.DataFrame(data = tsne_results_use, columns = ['tsne-use-one', 'tsne-use-two'])

    # Add category information for visualization
    tsne_use_df['top_level_category'] = df['top_level_category']

    print("\nT-SNE results DataFrame for USE Embeddings head:")
    display(tsne_use_df.head())

    # Visualize the t-SNE results
    plt.figure(figsize=(12, 10))

    sns.scatterplot(
        x="tsne-use-one", y="tsne-use-two",
        hue="top_level_category",
        palette=sns.color_palette("hsv", len(tsne_use_df['top_level_category'].unique())),
        data=tsne_use_df,
        legend="full",
        alpha=0.7,
        s=50
    )

    plt.title('T-SNE Visualization of USE Document Embeddings (Colored by Top Category)')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.legend(title='Top Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

else:
    print("\n'description' column not found in DataFrame. Cannot generate USE embeddings.")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# Apply K-Means clustering on the USE embeddings
# We use the number of unique top-level categories as the number of clusters
n_clusters = df['top_level_category'].nunique()
kmeans_use = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels_use = kmeans_use.fit_predict(use_embeddings_matrix)

print(f"\nApplying K-Means clustering with {n_clusters} clusters on USE embeddings...")

# To calculate the confusion matrix, we need to map the cluster labels to the true category labels.
# Since K-Means cluster labels are arbitrary (e.g., cluster 0 might correspond to 'Watches' or 'Baby Care'),
# we need to find the best mapping. A common approach is to assign each cluster to the most frequent
# true category within that cluster.

# Create a DataFrame to easily group by cluster and find the most frequent category
results_df = pd.DataFrame({'cluster': cluster_labels_use, 'true_category': df['top_level_category']})

# Find the most frequent true category for each cluster
cluster_to_category_mapping = results_df.groupby('cluster')['true_category'].agg(lambda x: x.mode()[0]).to_dict()

# Map the cluster labels to the predicted category labels using the mapping
predicted_categories = [cluster_to_category_mapping[label] for label in cluster_labels_use]

print("\nCalculating Confusion Matrix...")

# Calculate the confusion matrix
# The rows represent the true categories, and the columns represent the predicted categories.
cm = confusion_matrix(df['top_level_category'], predicted_categories)

# Get the unique true category labels for displaying on the confusion matrix
labels = df['top_level_category'].unique()
labels.sort() # Sort labels for consistent order

# Display the confusion matrix using ConfusionMatrixDisplay
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title('Confusion Matrix for K-Means Clustering on USE Embeddings')
plt.xlabel('Predicted Category')
plt.ylabel('True Category')
plt.tight_layout()
plt.show()

# Note: The confusion matrix shows how well the clustering aligns with the true categories.
# A perfect clustering would result in a diagonal matrix. Off-diagonal values indicate misclassifications.
# The mapping from cluster label to category is an approximation and might not be perfect,
# but the confusion matrix still provides a good overview of the clustering performance relative to the true labels.

# You can also calculate metrics like Adjusted Rand Index (ARI) or Normalized Mutual Information (NMI)
# to get a single score for the clustering performance. We already calculated ARI for other methods.
# For comparison, let's calculate ARI for USE as well (using the mapped predicted categories)
from sklearn.metrics import adjusted_rand_score
ari_use = adjusted_rand_score(df['top_level_category'], cluster_labels_use) # ARI compares the raw cluster labels, not the mapped ones
print(f"\nAdjusted Rand Index (ARI) score for K-Means on USE Embeddings: {ari_use:.4f}")

In [ ]:
!pip uninstall scikit-learn scipy -y
!pip install scikit-learn